In [2]:
import requests

URL = "https://realtime.gtfs.de/realtime-free.pb"

response = requests.get(URL, timeout=30)

print(response.status_code)
print(len(response.content), "bytes")


200
24501020 bytes


In [6]:
from google.transit import gtfs_realtime_pb2

feed = gtfs_realtime_pb2.FeedMessage()
feed.ParseFromString(response.content)

print(f"Number of entities: {len(feed.entity)}")


Number of entities: 98291


In [8]:
for entity in feed.entity[:10]:
    print(entity)

id: "971407tu"
trip_update {
  trip {
    trip_id: "971407"
    start_date: "20260904"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 0
    departure {
      delay: 60
      time: 1788525960
    }
    stop_id: "90601"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 1
    arrival {
      delay: 60
      time: 1788526440
    }
    departure {
      delay: 120
      time: 1788526560
    }
    stop_id: "677399"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 2
    arrival {
      delay: 120
      time: 1788527820
    }
    departure {
      delay: 120
      time: 1788527940
    }
    stop_id: "538819"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 3
    arrival {
      delay: 60
      time: 1788530340
    }
    departure {
      delay: 60
      time: 1788530520
    }
    stop_id: "10493"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_

In [11]:
import pandas as pd
from datetime import datetime

rows = []

for entity in feed.entity:
    if not entity.HasField("trip_update"):
        continue

    trip = entity.trip_update.trip

    for stop in entity.trip_update.stop_time_update:

        row = {
            "trip_id": trip.trip_id,
            "start_date": trip.start_date,
            "stop_id": stop.stop_id,
            "stop_sequence": stop.stop_sequence,
        }

        if stop.HasField("arrival"):
            row["arrival_time"] = datetime.fromtimestamp(
                stop.arrival.time
            )
            row["arrival_delay"] = stop.arrival.delay

        if stop.HasField("departure"):
            row["departure_time"] = datetime.fromtimestamp(
                stop.departure.time
            )
            row["departure_delay"] = stop.departure.delay

        rows.append(row)

df = pd.DataFrame(rows)

df.head(100)

,trip_id,start_date,stop_id,stop_sequence,departure_time,departure_delay,arrival_time,arrival_delay
0,971407,20260904,90601,0,2026-09-04 14:46:00,60.0,NaT,NaN
1,971407,20260904,677399,1,2026-09-04 14:56:00,120.0,2026-09-04 14:54:00,60.0
2,971407,20260904,538819,2,2026-09-04 15:19:00,120.0,2026-09-04 15:17:00,120.0
3,971407,20260904,10493,3,2026-09-04 16:02:00,60.0,2026-09-04 15:59:00,60.0
4,971407,20260904,457801,4,2026-09-04 16:49:00,0.0,2026-09-04 16:45:00,120.0
...,...,...,...,...,...,...,...,...
95,861193,20260904,597459,31,NaT,NaN,2026-09-04 19:55:00,0.0
96,1196166,20260904,319386,0,2026-09-04 21:45:00,0.0,NaT,NaN
97,1390148,20260904,347017,0,2026-09-04 20:29:00,0.0,NaT,NaN
98,875521,20260904,476674,0,2026-09-04 22:00:00,0.0,NaT,NaN


In [12]:
df["arrival_time"].min()


Timestamp('2026-09-04 10:14:00')